# Step 1 — Download NOAA Local Climatological Data (LCD)

Downloads hourly LCD records (routine METAR `FM-15` and special reports `FM-16`) for the airports in the study area, 2018–2022.

- **Station list:** NOAA `isd-history.csv` (downloaded automatically to `data/raw/`); stations are matched to airports by ICAO code.
- **Output:** `data/raw/noaa_lcd/<year>/<STATE>_<IATA>_<year>.csv`
- Files that already exist are skipped, so the notebook can be re-run after an interruption.

Source: <https://www.ncei.noaa.gov/data/local-climatological-data/>

In [2]:
import os
import time
import urllib.request
import urllib.error
import pandas as pd
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'README.md').exists())  # project root

IATA_AIRPORTS = [
    'ABY', 'ATL', 'BUF', 'DFW', 'LGA', 'CHO', 'EWN', 'DCA', 'FAY', 'OAJ', 'RDU', 'SRQ',
    'AUS', 'CSG', 'PIA', 'IAD', 'GSO', 'AVL', 'ORD', 'ROC', 'JAX', 'VLD', 'IAH', 'SAV',
    'ROA', 'SYR', 'SAT', 'BMI', 'BQK', 'ORF', 'GNV', 'TPA', 'CLT', 'ILM', 'ELM', 'MCO',
    'RIC', 'TLH', 'GRK', 'HRL', 'HPN', 'RSW', 'PBI', 'FLL', 'JFK', 'SJC', 'LGB', 'LAX',
    'SFO', 'HOU', 'DEN', 'SEA', 'SAN', 'ALB', 'OAK', 'PSP', 'SMF', 'BUR', 'SWF', 'DAB',
    'ELP', 'PNS', 'COS', 'CLL', 'ECP', 'MFE', 'AGS', 'VPS', 'EYW', 'PHF', 'MLI', 'SFB',
    'SCK', 'FAT', 'BLI', 'PGD', 'PIE', 'PBG', 'OGS', 'USA', 'SPI', 'MTJ', 'IAG', 'BLV',
    'RFD', 'PSC', 'MRY', 'GJT', 'LRD', 'SMX', 'ONT', 'ACV', 'BFL', 'ASE', 'GEG', 'DRO',
    'SBA', 'RDD', 'SNA', 'MMH', 'SBP', 'MAF', 'STS', 'BRO', 'HDN', 'ITH', 'BGM', 'DAL',
    'MDW', 'PUB', 'UIN', 'EGE', 'AMA', 'ISP', 'MIA', 'CRP', 'LBB', 'MLB', 'GUC', 'CMI',
    'PUW', 'EAT', 'ALW', 'YKM', 'TYR', 'SJT', 'BPT', 'ACT', 'SHD', 'ABI', 'PGV', 'LYH',
    'GGG', 'ART', 'SPS', 'DRT', 'PAE', 'DEC', 'VCT', 'ALS', 'BIH'
]

TARGET_ICAO = ['K' + code for code in IATA_AIRPORTS]
YEARS = [2018, 2019, 2020, 2021, 2022]

USER_AGENT = "classifier-test-noaa-fetch/1.0 (glodfrog213@gmail.com)"
REQUEST_TIMEOUT = 30


def http_get(url, timeout=REQUEST_TIMEOUT):
    req = urllib.request.Request(url, headers={"User-Agent": USER_AGENT})
    return urllib.request.urlopen(req, timeout=timeout)


def download_file(url, save_path, max_retries=3):
    """Download with User-Agent and timeout; write to .tmp, then rename."""
    if os.path.exists(save_path):
        return True

    tmp_path = save_path + ".tmp"
    for attempt in range(max_retries):
        try:
            with http_get(url) as resp, open(tmp_path, "wb") as f:
                while True:
                    chunk = resp.read(64 * 1024)
                    if not chunk:
                        break
                    f.write(chunk)
            os.rename(tmp_path, save_path)
            time.sleep(0.3)
            return True
        except urllib.error.HTTPError as e:
            _cleanup(tmp_path)
            if e.code == 404:
                return False
            print(f"HTTP error {e.code} at {url}; retrying...")
            time.sleep(2 * (attempt + 1))
        except Exception as e:
            _cleanup(tmp_path)
            print(f"Network error: {e}; retrying...")
            time.sleep(3 * (attempt + 1))

    return False


def _cleanup(path):
    try:
        if os.path.exists(path):
            os.remove(path)
    except OSError:
        pass


def get_target_stations():
    """Download the NOAA station list and keep the target airports."""
    print("Downloading NOAA station list (isd-history.csv)...")
    url_history = "https://www.ncei.noaa.gov/pub/data/noaa/isd-history.csv"

    tmp_csv = str(ROOT / "data" / "raw" / "isd-history.csv")
    os.makedirs(os.path.dirname(tmp_csv), exist_ok=True)
    if not os.path.exists(tmp_csv):
        with http_get(url_history) as resp, open(tmp_csv, "wb") as f:
            f.write(resp.read())

    df = pd.read_csv(tmp_csv, dtype={'USAF': str, 'WBAN': str})

    df = df[
        (df['CTRY'] == 'US') &
        (df['ICAO'].isin(TARGET_ICAO)) &
        (df['END'].astype(str) >= '20221201')
    ].copy()

    df['STATE'] = df['STATE'].fillna('XX')
    df['NOAA_ID'] = df['USAF'].str.zfill(6) + df['WBAN'].str.zfill(5)

    # If an ICAO code maps to several stations, keep the most recent one.
    df = df.sort_values('END', ascending=False).drop_duplicates(subset=['ICAO'], keep='first')

    print(f"-> Matched {len(df)} weather stations to target airports.")
    missing = sorted(set(TARGET_ICAO) - set(df['ICAO']))
    if missing:
        print(f"-> {len(missing)} ICAO codes without a matching station: {', '.join(missing)}")
    return df


def main():
    stations_df = get_target_stations()

    base_dir = str(ROOT / "data" / "raw" / "noaa_lcd")
    os.makedirs(base_dir, exist_ok=True)

    for year in YEARS:
        year_dir = os.path.join(base_dir, str(year))
        os.makedirs(year_dir, exist_ok=True)
        print(f"\n===== Year {year} =====")

        for _, row in stations_df.iterrows():
            station_id = row['NOAA_ID']
            icao = row['ICAO']
            iata = icao[1:]
            state = row['STATE']

            csv_url = (
                f"https://www.ncei.noaa.gov/data/local-climatological-data/"
                f"access/{year}/{station_id}.csv"
            )

            file_name = f"{state}_{iata}_{year}.csv"
            save_path = os.path.join(year_dir, file_name)

            print(f"{file_name}...", end=" ", flush=True)
            if download_file(csv_url, save_path):
                print("ok")
            else:
                print("no data")


if __name__ == "__main__":
    main()